# Assignment 3: Evaluate a small RAG system (City Library FAQ)

**Submit code and generated results only. No theory answers or written report.**

You get `data/assignment_rag.json` with 3 library documents (`documents`) and 8 questions (`queries`).
Each query has: `query`, `doc_id`, `answer_span` (the exact words that answer it), `reference_answer`, and `answer_keywords` (any one of them in an answer means it is correct).

1. **Chunks and relevance.** Write `make_chunks(docs, size, overlap)` (count words, not tokens) and `grade(chunk, query)`: return 2 if the chunk contains the `answer_span` (ignore upper/lower case and extra spaces), 1 if it is from the same `doc_id` but has no answer span, otherwise 0. Check that every `answer_span` appears in its own document. Print how many chunks you get with `size=30, overlap=0`.
2. **Retrieval metrics.** Write `hit_at_k`, `reciprocal_rank`, and `ndcg_at_k` for a list of grades in ranked order (a grade of 2 or more is relevant; use `k=3`). Test them on `[0, 2, 0, 1]`. Embed the chunks and queries with `embed()` from `common.py`, rank chunks by cosine similarity, and print the average Hit@3, MRR@3, and NDCG@3 for `size=30, overlap=0`.
3. **Grid search.** Try `size` in `[30, 60]` and `overlap` in `[0, 15]`. Print a pandas table with the number of chunks and the three metrics for each setting, and print the best setting (highest Hit@3, ties broken by MRR@3).
4. **Generation.** With the best setting, retrieve the top 3 chunks for each query and ask Gemini to answer using only those chunks. Add these columns to a pandas table: `retrieval_hit` (is the answer span in the top 3?), `answer_correct` (code check with `answer_keywords`), and `faithful` (LLM judge returning JSON with `answer` Pass/Fail and `reasoning`). Add one extra row with a **clearly marked fixture**: an intentionally unfaithful answer that you write yourself. Print the table and the counts of each diagnosis: `OK`, `generation problem` (hit but wrong), `retrieval problem` (no hit and wrong), `lucky` (no hit but correct).
5. **Context swap.** Pick one query. Answer it with its real document, then with an unrelated document, using (a) a plain system prompt and (b) a system prompt that says to use only the documents and to say "I don't know" otherwise. Print the 4 answers side by side.

**Hints:**
- Reuse `ask()` and `embed()` from `common.py`, and ideas from Notebooks 2, 3, 5 and 7 (Chapter 7 notebook).
- Normalize text with `" ".join(text.lower().split())` before checking `answer_span`.
- Embed all texts in one `embed(list_of_texts)` call; make unit-length vectors so that cosine similarity is a dot product.
- `ndcg_at_k`: DCG = sum of `grade / log2(rank + 1)` over the top k; the ideal DCG uses the same top-k grades sorted from best to worst; return 0 if the ideal DCG is 0. For `[0, 2, 0, 1]` with `k=3` you should get RR = 0.5 and NDCG about 0.63.
- Do not combine `temperature` with `as_json=True`; use `temperature=0` only for the answering call.
- Validate the judge output: `answer` must be exactly `Pass` or `Fail`.

**Deliverables:** your solution notebook and the printed results. Keep the API key in `.env`.

In [ ]:
# Starter cell: add your solution cells below this one.
from common import ask, embed, MODEL
import json, math, re
import numpy as np
import pandas as pd
from IPython.display import display

data = json.load(open("data/assignment_rag.json", encoding="utf-8"))
docs, queries = data["documents"], data["queries"]
K = 3
print("Model:", MODEL, "|", len(docs), "documents |", len(queries), "queries")